In [ ]:
# !pip install gymnasium highway-env torch numpy matplotlib pandas

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
import highway_env

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from base64 import b64encode

In [ ]:
try:
    highway_env.register_highway_envs()
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
config = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 5,
        "features": ["presence", "x", "y", "vx", "vy"],
        "normalize": True,
        "absolute": False
    },
    "action": {
        "type": "DiscreteMetaAction"
    },
    "lanes_count": 4,
    "vehicles_count": 30,
    "duration": 40,
    "simulation_frequency": 15,
    "policy_frequency": 5,
    "collision_reward": -1.0,
    "right_lane_reward": 0.1,
    "high_speed_reward": 0.4,
    "lane_change_reward": 0.0,
    "reward_speed_range": [20, 30],
    "normalize_reward": True
}

In [ ]:
env = gym.make("highway-v0", config=config, render_mode="rgb_array")

state, info = env.reset()

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial state shape:", np.array(state).shape)

env.close()

In [ ]:
def preprocess_state(state):
    return np.asarray(state, dtype=np.float32).flatten()


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


def moving_average(values, window=20):
    if len(values) < window:
        return values

    return pd.Series(values).rolling(window=window).mean().to_numpy()

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.dones = []
        self.values = []

    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.log_probs.clear()
        self.rewards.clear()
        self.dones.clear()
        self.values.clear()

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()

        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, action_dim)
        )

        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def get_distribution(self, state):
        logits = self.actor(state)
        return Categorical(logits=logits)

    def get_action_and_value(self, state, action=None):
        distribution = self.get_distribution(state)

        if action is None:
            action = distribution.sample()

        log_prob = distribution.log_prob(action)
        entropy = distribution.entropy()
        value = self.critic(state).squeeze(-1)

        return action, log_prob, entropy, value

    def act_deterministic(self, state):
        logits = self.actor(state)
        return torch.argmax(logits, dim=-1)

In [ ]:
class PPOAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        gamma=0.99,
        gae_lambda=0.95,
        clip_epsilon=0.2,
        learning_rate=3e-4,
        update_epochs=8,
        minibatch_size=64,
        value_coef=0.5,
        entropy_coef=0.01,
        max_grad_norm=0.5
    ):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.update_epochs = update_epochs
        self.minibatch_size = minibatch_size
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

        self.policy = ActorCritic(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)

        self.buffer = RolloutBuffer()

    def select_action(self, state):
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        with torch.no_grad():
            action, log_prob, _, value = self.policy.get_action_and_value(state_tensor)

        action_item = action.item()

        self.buffer.states.append(state)
        self.buffer.actions.append(action_item)
        self.buffer.log_probs.append(log_prob.item())
        self.buffer.values.append(value.item())

        return action_item

    def compute_returns_and_advantages(self, next_value=0.0):
        rewards = np.array(self.buffer.rewards, dtype=np.float32)
        dones = np.array(self.buffer.dones, dtype=np.float32)
        values = np.array(self.buffer.values, dtype=np.float32)

        advantages = np.zeros_like(rewards, dtype=np.float32)
        gae = 0.0

        for step in reversed(range(len(rewards))):
            if step == len(rewards) - 1:
                next_non_terminal = 1.0 - dones[step]
                next_value_step = next_value
            else:
                next_non_terminal = 1.0 - dones[step]
                next_value_step = values[step + 1]

            delta = rewards[step] + self.gamma * next_value_step * next_non_terminal - values[step]
            gae = delta + self.gamma * self.gae_lambda * next_non_terminal * gae
            advantages[step] = gae

        returns = advantages + values

        return returns, advantages

    def update(self, next_value=0.0):
        returns, advantages = self.compute_returns_and_advantages(next_value)

        states = torch.tensor(np.array(self.buffer.states), dtype=torch.float32, device=device)
        actions = torch.tensor(np.array(self.buffer.actions), dtype=torch.long, device=device)
        old_log_probs = torch.tensor(np.array(self.buffer.log_probs), dtype=torch.float32, device=device)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)
        advantages = torch.tensor(advantages, dtype=torch.float32, device=device)

        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        dataset_size = states.shape[0]
        indices = np.arange(dataset_size)

        total_policy_loss = 0.0
        total_value_loss = 0.0
        total_entropy = 0.0
        update_count = 0

        for _ in range(self.update_epochs):
            np.random.shuffle(indices)

            for start in range(0, dataset_size, self.minibatch_size):
                end = start + self.minibatch_size
                batch_indices = indices[start:end]

                batch_states = states[batch_indices]
                batch_actions = actions[batch_indices]
                batch_old_log_probs = old_log_probs[batch_indices]
                batch_returns = returns[batch_indices]
                batch_advantages = advantages[batch_indices]

                _, new_log_probs, entropy, values = self.policy.get_action_and_value(
                    batch_states,
                    batch_actions
                )

                ratio = torch.exp(new_log_probs - batch_old_log_probs)

                unclipped_objective = ratio * batch_advantages
                clipped_objective = torch.clamp(
                    ratio,
                    1.0 - self.clip_epsilon,
                    1.0 + self.clip_epsilon
                ) * batch_advantages

                policy_loss = -torch.min(unclipped_objective, clipped_objective).mean()
                value_loss = nn.MSELoss()(values, batch_returns)
                entropy_loss = entropy.mean()

                loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
                self.optimizer.step()

                total_policy_loss += policy_loss.item()
                total_value_loss += value_loss.item()
                total_entropy += entropy_loss.item()
                update_count += 1

        self.buffer.clear()

        return {
            "policy_loss": total_policy_loss / update_count,
            "value_loss": total_value_loss / update_count,
            "entropy": total_entropy / update_count
        }

In [ ]:
def train_ppo_highway(
    total_timesteps=50_000,
    rollout_steps=1024,
    seed=42,
    print_every=10
):
    set_seed(seed)

    env = gym.make("highway-v0", config=config, render_mode="rgb_array")

    state, info = env.reset(seed=seed)
    state = preprocess_state(state)

    state_dim = state.shape[0]
    action_dim = env.action_space.n

    agent = PPOAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        gamma=0.99,
        gae_lambda=0.95,
        clip_epsilon=0.2,
        learning_rate=3e-4,
        update_epochs=8,
        minibatch_size=64,
        value_coef=0.5,
        entropy_coef=0.01,
        max_grad_norm=0.5
    )

    episode_rewards = []
    episode_lengths = []
    policy_losses = []
    value_losses = []
    entropies = []

    episode_reward = 0.0
    episode_length = 0
    episode = 0
    timestep = 0
    done = False

    while timestep < total_timesteps:
        for _ in range(rollout_steps):
            action = agent.select_action(state)

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            next_state = preprocess_state(next_state)

            agent.buffer.rewards.append(float(reward))
            agent.buffer.dones.append(float(done))

            episode_reward += reward
            episode_length += 1
            timestep += 1

            state = next_state

            if done:
                episode += 1
                episode_rewards.append(episode_reward)
                episode_lengths.append(episode_length)

                if episode % print_every == 0:
                    recent_mean = np.mean(episode_rewards[-print_every:])
                    print(
                        f"Episode: {episode} | "
                        f"Timestep: {timestep} | "
                        f"Recent mean reward: {recent_mean:.3f} | "
                        f"Last reward: {episode_reward:.3f}"
                    )

                state, info = env.reset()
                state = preprocess_state(state)

                episode_reward = 0.0
                episode_length = 0
                done = False

            if timestep >= total_timesteps:
                break

        if done:
            next_value = 0.0
        else:
            state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                next_value = agent.policy.critic(state_tensor).squeeze(-1).item()

        losses = agent.update(next_value=next_value)

        policy_losses.append(losses["policy_loss"])
        value_losses.append(losses["value_loss"])
        entropies.append(losses["entropy"])

    env.close()

    results = {
        "episode_rewards": episode_rewards,
        "episode_lengths": episode_lengths,
        "policy_losses": policy_losses,
        "value_losses": value_losses,
        "entropies": entropies
    }

    return agent, results

In [ ]:
agent, results = train_ppo_highway(
    total_timesteps=50_000,
    rollout_steps=1024,
    seed=42,
    print_every=10
)

In [ ]:
# agent, results = train_ppo_highway(
#     total_timesteps=200_000,
#     rollout_steps=2048,
#     seed=42,
#     print_every=10
# )

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(results["episode_rewards"], label="Episode Reward")

if len(results["episode_rewards"]) >= 20:
    plt.plot(moving_average(results["episode_rewards"], 20), label="20-Episode Moving Average")

plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("PPO on highway-v0: Episode Reward")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(results["episode_lengths"], label="Episode Length")

if len(results["episode_lengths"]) >= 20:
    plt.plot(moving_average(results["episode_lengths"], 20), label="20-Episode Moving Average")

plt.xlabel("Episode")
plt.ylabel("Steps")
plt.title("PPO on highway-v0: Episode Length")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(results["policy_losses"], label="Policy Loss")
plt.xlabel("PPO Update")
plt.ylabel("Loss")
plt.title("PPO Policy Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(results["value_losses"], label="Value Loss")
plt.xlabel("PPO Update")
plt.ylabel("Loss")
plt.title("PPO Value Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def evaluate_ppo(agent, episodes=10, seed=100):
    env = gym.make("highway-v0", config=config, render_mode="rgb_array")

    rewards = []
    lengths = []

    for ep in range(episodes):
        state, info = env.reset(seed=seed + ep)
        state = preprocess_state(state)

        done = False
        total_reward = 0.0
        steps = 0

        while not done:
            state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

            with torch.no_grad():
                action = agent.policy.act_deterministic(state_tensor).item()

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            state = preprocess_state(next_state)
            total_reward += reward
            steps += 1

        rewards.append(total_reward)
        lengths.append(steps)

    env.close()

    print("Evaluation episodes:", episodes)
    print("Mean reward:", np.mean(rewards))
    print("Reward std:", np.std(rewards))
    print("Best reward:", np.max(rewards))
    print("Worst reward:", np.min(rewards))
    print("Mean episode length:", np.mean(lengths))

    return rewards, lengths

In [ ]:
eval_rewards, eval_lengths = evaluate_ppo(agent, episodes=10)

In [ ]:
torch.save(agent.policy.state_dict(), "ppo_highway_discrete.pth")

In [ ]:
training_df = pd.DataFrame({
    "episode": np.arange(1, len(results["episode_rewards"]) + 1),
    "reward": results["episode_rewards"],
    "length": results["episode_lengths"]
})

training_df.to_csv("ppo_highway_training_results.csv", index=False)
training_df.head()

In [ ]:
summary = {
    "Mean training reward last 20": np.mean(results["episode_rewards"][-20:]) if len(results["episode_rewards"]) >= 20 else np.mean(results["episode_rewards"]),
    "Best training reward": np.max(results["episode_rewards"]),
    "Mean evaluation reward": np.mean(eval_rewards),
    "Evaluation reward std": np.std(eval_rewards),
    "Best evaluation reward": np.max(eval_rewards),
    "Mean evaluation length": np.mean(eval_lengths)
}

summary

In [ ]:
def load_saved_agent(path="ppo_highway_discrete.pth"):
    env = gym.make("highway-v0", config=config, render_mode="rgb_array")

    state, info = env.reset()
    state = preprocess_state(state)

    state_dim = state.shape[0]
    action_dim = env.action_space.n

    loaded_agent = PPOAgent(
        state_dim=state_dim,
        action_dim=action_dim
    )

    loaded_agent.policy.load_state_dict(torch.load(path, map_location=device))
    loaded_agent.policy.eval()

    env.close()

    return loaded_agent

In [ ]:
loaded_agent = load_saved_agent("ppo_highway_discrete.pth")

In [ ]:
def record_video(agent, video_folder="ppo_highway_video"):
    os.makedirs(video_folder, exist_ok=True)

    env = gym.make("highway-v0", config=config, render_mode="rgb_array")

    env = RecordVideo(
        env,
        video_folder=video_folder,
        episode_trigger=lambda episode_id: True,
        name_prefix="ppo_highway"
    )

    state, info = env.reset()
    state = preprocess_state(state)

    done = False
    total_reward = 0.0

    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        with torch.no_grad():
            action = agent.policy.act_deterministic(state_tensor).item()

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        state = preprocess_state(next_state)
        total_reward += reward

    env.close()

    print("Video saved to:", video_folder)
    print("Episode reward:", total_reward)

In [ ]:
record_video(agent)

In [ ]:
def show_video(video_folder="ppo_highway_video"):
    mp4_files = []

    for root, dirs, files in os.walk(video_folder):
        for file in files:
            if file.endswith(".mp4"):
                mp4_files.append(os.path.join(root, file))

    if len(mp4_files) == 0:
        print("No video found.")
        return

    video_path = mp4_files[0]
    print("Showing:", video_path)

    mp4 = open(video_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    return HTML(f"""
    <video width="640" controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)

In [ ]:
show_video()